# QuantumCylinder Final Submission Notebook

**Contribution.** QuantumCylinder compares random-unitary diffusion and Hamiltonian projected diffusion under the same fidelity-based MMD/Wasserstein-type metrics, then studies measurement-induced projected denoising through the trade-off among recoverability, post-selection success probability, diversity retention, and control/resource cost.

This notebook is the judge-facing final answer. Development logs, automation notes, and local machine paths are intentionally omitted.

## 1. Problem-to-Solution Map

| Problem part | What this notebook answers | Final evidence |
| --- | --- | --- |
| 1(a) | Construct a 2-qubit target ensemble near `|00>`. | Target fidelity sanity table. |
| 1(b) | Define fidelity-based MMD and Wasserstein-type metrics. | `MMD(S0,S0)=0`, `W(S0,S0)=0`, one-step scramble nonzero. |
| 1(c) | Show random-unitary forward diffusion. | Random-unitary trajectory with a Haar-random reference baseline. |
| 2(a)-(c) | Build fixed Hamiltonian projected diffusion with a complement qubit. | Hamiltonian distance curve and projection explanation. |
| 2(d) | Compare resource/control cost at comparable diffusion strength. | Metric-aligned comparison, not shared raw x-axis. |
| 3(a) | Demonstrate a simple measurement-induced denoising step. | Continuous post-selection before/after metrics. |
| 3(b) | Analyze a controlled measurement-basis trade-off. | Gain/success/diversity/collapse-defense table. |
| 3(c) | Propose an analysis-guided improvement. | Two-way Hamiltonian post-selection comparison. |


## 2. Problem 1(a)(b): Target Ensemble and Fidelity-Based Metrics

The target ensemble `S0` is generated around `|00>` with `N=80`, `sigma=0.10`, seed `7`. We use fidelity-based MMD and an infidelity-cost Wasserstein-type distance for every later comparison.

**Main Table 1: target and metric sanity checks**

| Item | Value |
| --- | ---: |
| N | `80` |
| sigma | `0.1` |
| seed | `7` |
| mean fidelity to `|00>` | `0.995692` |
| min fidelity to `|00>` | `0.972637` |
| max fidelity to `|00>` | `0.999970` |
| std fidelity to `|00>` | `0.004599` |
| `MMD(S0,S0)` | `0.000000000000` |
| `Wasserstein(S0,S0)` | `0.000000000000` |
| `MMD(S1_random_unitary,S0)` | `0.882606` |
| `Wasserstein(S1_random_unitary,S0)` | `0.733305` |

Source files: `tables/problem_1b_metric_diagnostics.md` and `tables/problem_1b_metric_diagnostics.csv`.


## 3. Problem 1(c): Random-Unitary Diffusion with Haar Reference

Random-unitary diffusion applies random local rotations and an entangling gate in layers. The initial target ensemble is clustered near `|00>`. Under the chosen random-unitary scrambling layer, the MMD and Wasserstein-type distances from `S0` increase rapidly and approach the Haar-random reference level. This indicates that the original cluster structure is destroyed under strong scrambling. The saturation behavior should be interpreted as a strong-scrambling baseline rather than a slow diffusion schedule.

The Haar reference is not a trained target distribution. It is a reference distance `D(S0, S_Haar)` computed from 64 Haar-random two-qubit ensembles with the same ensemble size `N=80` as `S0`.

| Metric | Haar mean | Haar std | Final `k=12` distance |
| --- | ---: | ---: | ---: |
| MMD | `0.869583` | `0.024043` | `0.828093` |
| Wasserstein-type | `0.724439` | `0.021491` | `0.686108` |

**Main Figure 1: random-unitary diffusion approaches a Haar-like distance plateau**

![Random-unitary Haar baseline](figures/fig2_random_unitary_haar_baseline.png)

Detailed random-unitary rows are copied to `tables/problem_1c_random_unitary_metrics.csv`; the Haar summary is copied to `tables/problem1_haar_reference.csv`.


## 4. Problem 2(a)(b)(c): Hamiltonian Projected Diffusion

For Problem 2, the data system `M` is augmented with a complement qubit `F`. The combined system evolves under a fixed three-qubit Hamiltonian, and then `F` is projected to create a data-system ensemble.

Important distinction: the random-unitary layer index `k` and Hamiltonian time `t` are different native controls. We do not compare equal x-axis positions as if they were the same physical resource.

Random-unitary diffusion rapidly approaches a Haar-like reference level under strong scrambling. Hamiltonian projected diffusion, by contrast, uses a fixed Hamiltonian and projection onto the data system, so its distance curves can show schedule-dependent fluctuations or partial saturation. Therefore, the comparison is not simply about which distance is larger; it is about diffusion strength, fixed-control structure, schedule sensitivity, and control overhead.

Observed Hamiltonian projected diffusion summary:

- Max Hamiltonian MMD: `1.249244` at `t=1.000000`.
- Max Hamiltonian Wasserstein-type distance: `0.883968` at `t=4.000000`.

**Main Figure 2: native diffusion curves**

![Distance curves](figures/problem_1_2_distance_curves.png)

Detailed Hamiltonian rows are copied to `tables/problem_2_hamiltonian_metrics.csv`.


## 5. Problem 2(d): Resource / Control-Cost Comparison

Problem 2(d) is reported at comparable output metric strength, not equal raw parameter value. Random-unitary diffusion is controlled by layer count, random single-qubit rotations, entangling operation count, and fine-grained random gate control. Hamiltonian projected diffusion is controlled by total evolution time, a fixed Hamiltonian, and measurement/projection basis. We do not claim Hamiltonian projected diffusion is always better; the point is the trade-off between random gate-level control and fixed-Hamiltonian projected evolution.

**Main Figure 3: metric-aligned comparison**

![Metric aligned comparison](figures/problem_1_2_metric_aligned_comparison.png)

**Main Table 2: comparable-strength resource/control proxy**

| Matched by | Random step | Hamiltonian time | Metric gap | Random controls | Entanglers | Hamiltonian fixed terms | Interpretation |
| --- | --- | --- | --- | --- | --- | --- | --- |
| mmd | 1 | 0.333333 | 0.002259 | 6 | 1 | 8 | compare at similar output metric, not equal x-axis parameter |
| wasserstein | 7 | 3.333333 | 0.001220 | 42 | 7 | 8 | compare at similar output metric, not equal x-axis parameter |

Source file: `tables/problem_2d_resource_matches.csv`.


## 6. Problem 3(a): Simple Measurement-Induced Denoising Step

Problem 3(a) is not a full trainable QuDDPM. It is a small measurement-induced denoising proxy.

The combined `M+F` system evolves unitarily, but measuring the complement qubit `F` in a chosen basis and post-selecting an outcome induces an effective non-unitary map on the data system `M`:

```text
|phi_i_m(t,b)> = (I_M tensor <b_m|) exp(-i H t) (|psi_i>_M tensor |0>_F) / sqrt(p_i_m)
```

Low-probability post-selection candidates are not replaced by `|00>` or by the original state; they are excluded before scoring.

In the 20-seed summary, the continuous post-selection reference has median MMD gain `0.097056`, median Wasserstein gain `0.147983`, diversity retention `0.823217`, and success probability `0.468122`.

**Main Figure 4: simple denoising before/after result**

![Problem 3 denoising improvement](figures/problem_3a_denoising_improvement.png)


## 7. Problem 3(b): Measurement Basis Controls the Recoverability-Success-Diversity Trade-Off

Measurement basis is the controlled modification. In the Hamiltonian projected setting, the combined data-plus-complement system `M+F` evolves unitarily:

```text
|Psi_i(t)> = exp(-i H t) (|psi_i>_M tensor |0>_F)
```

For a complement-basis vector `|b_m>`, post-selection gives

```text
|phi_i_m(t,b)> = (I_M tensor <b_m|) |Psi_i(t)> / sqrt(p_i_m)
p_i_m = ||(I_M tensor <b_m|) |Psi_i(t)>||^2
```

The full `M+F` evolution is unitary, but conditioning on a measurement outcome gives an effective non-unitary map on `M`. Therefore, changing the measurement basis changes the direction and strength of the effective projected map.

Axis-only `Z/X/Y` projection is the simplest interpretable Pauli-basis baseline. Continuous post-selection generalizes this baseline to Bloch-sphere measurement directions `(theta, phi)`. The goal is not to claim continuous bases overwhelmingly beat axis-only bases; the goal is to evaluate the trade-off among distance gain/recoverability, post-selection success probability, and ensemble diversity retention.

The observed axis-only margin is small: median score margin `0.010000`, with `18 / 120` nonpositive axis-margin rows. Distance-only metrics are insufficient because a lower distance can come from a low-success event or from collapsing the ensemble diversity.

**Main Table 3: Problem 3(b) measurement-basis trade-off**

| Method | MMD gain | Wasserstein gain | Success probability | Diversity retention | Interpretation |
| --- | --- | --- | --- | --- | --- |
| identity/no-denoising random-unitary input | 0.000000 | 0.000000 | 1.000000 | 1.000000 | identity reference; no reverse denoising step |
| best exact Z/X/Y axis projection | 0.086055 | 0.142594 | 0.465058 | 0.810592 | interpretable Z/X/Y Pauli-basis baseline |
| continuous measurement-basis post-selection | 0.097056 | 0.147983 | 0.468122 | 0.823217 | continuous Bloch-sphere measurement-basis control; small margin over axis-only |
| diagnostic collapse-to-target-centroid filter | 0.859292 | 0.714276 | 1.000000 | 0.000000 | diagnostic failure mode: distance improves while diversity collapses |

Conclusion: **The main conclusion is not that continuous measurement bases overwhelmingly outperform axis-only bases. The observed axis-only margin is small. The important observation is that projected denoising must be evaluated by a trade-off: larger distance reduction can come at the cost of lower post-selection success probability or reduced ensemble diversity.**

Source file: `tables/problem3b_measurement_basis_tradeoff.csv`.


## 8. Problem 3(c): Analysis-Guided Improvement from 3-b

Because 3-b shows that stronger measurement-induced contraction can improve distance metrics but may reduce success probability, we test a stronger but more costly projected denoising step: Hamiltonian two-way post-selection.

Hamiltonian two-way post-selection applies the measurement-induced projected denoising idea more strongly by applying the post-selected contraction twice. It is an analysis-guided 3(c) trade-off candidate, not an unconditional win.

**Main Table 4: two-way projected denoising comparison**

| Method | MMD gain | Wasserstein gain | Diversity retention | Success probability | Role / interpretation |
| --- | --- | --- | --- | --- | --- |
| one-way continuous post-selection reference | 0.056388 | 0.120620 | 0.848836 | 0.467554 | 3-b reference level for one-way measurement-basis post-selection |
| Hamiltonian + random final kick | 0.056695 | 0.119401 | 0.848403 | 0.467554 | ablation only; random correction is not the main improvement |
| Hamiltonian two-way post-selection | 0.101374 | 0.136426 | 0.829273 | 0.227065 | 3-c main trade-off candidate: stronger distance gain, lower success probability |

Conclusion: **Hamiltonian two-way post-selection gives stronger distance improvement but sacrifices post-selection success probability. This follows directly from the 3-b trade-off analysis.**

Actor-critic is target-aware because its reward uses the raw target ensemble. It is therefore a toy policy-search candidate under target prior, not a general unknown-target denoiser. Its supporting row is kept in `tables/problem_3_appendix_rows.csv`, not as the main 3(c) conclusion.

Source file: `tables/problem3c_analysis_guided_improvement.csv`.


## 9. Limitations and What We Do Not Claim

- We do not claim quantum advantage.
- We do not claim hardware advantage.
- We do not claim a full trainable QuDDPM reverse process.
- We do not claim continuous measurement bases always or strongly beat axis-only Pauli bases.
- We do not treat the collapse-to-centroid diagnostic as a physical denoiser.
- We do not present actor-critic as a general unknown-target denoiser because it uses target information in the reward.

## 10. Reproducibility Commands

Run from the repository root:

```powershell
python -m pytest
python submission/run_all.py --quick
python scripts/create_solution_haar_baseline.py
python scripts/summarize_problem_3_seed_sweep.py
python scripts/run_problem_3_hamiltonian_variant_candidates.py
python scripts/summarize_problem_3_method_portfolio.py
```

The final-facing artifacts used by this notebook are copied into `solution/figures/` and `solution/tables/`.


In [ ]:
from pathlib import Path

base = Path.cwd()
if not (base / 'figures').exists() and (base / 'solution' / 'figures').exists():
    base = base / 'solution'

required = [
    'figures/fig2_random_unitary_haar_baseline.png',
    'figures/problem_1_2_distance_curves.png',
    'figures/problem_1_2_metric_aligned_comparison.png',
    'figures/problem_3a_denoising_improvement.png',
    'figures/problem_3c_hamiltonian_variant_summary.png',
    'tables/problem_1b_metric_diagnostics.md',
    'tables/problem_1b_metric_diagnostics.csv',
    'tables/problem1_haar_reference.csv',
    'tables/problem_2d_resource_matches.csv',
    'tables/problem3b_measurement_basis_tradeoff.csv',
    'tables/problem3c_analysis_guided_improvement.csv',
]
missing = [p for p in required if not (base / p).exists()]
if missing:
    raise FileNotFoundError(missing)
print(f'All final solution artifacts are present under {base}.')
